# 07b IQ Blobs (Raw ADC) – Load & Analyze

Loads `ds_raw` from a local `.nc` file and runs post-processing (IQ blobs, trajectories, SNR). Run notebook 01 first to execute and save.

## Imports

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from iqcc_calibration_tools.qualibrate_config.qualibrate.node import QualibrationNode
from quam_builder.architecture.superconducting.qpu import FluxTunableQuam as Quam
from calibration_utils.iq_blobs_raw_adc import (
    Parameters,
    plot_raw_adc_traces,
    plot_iq_trajectories,
    process_raw_dataset,
    fit_raw_data,
    fit_snr_with_gaussians,
    plot_iq_blobs,
)
from qualibration_libs.parameters import get_qubits

## Node initialisation

In [ ]:
description = """
IQ BLOBS (RAW ADC)
Uses raw ADC traces instead of on-board measure and demodulation.
"""

node = QualibrationNode[Parameters, Quam](
    name="07b_iq_blobs_raw_adc",
    description=description,
    parameters=Parameters(),
)

@node.run_action(skip_if=node.modes.external)
def custom_param(node: QualibrationNode[Parameters, Quam]):
    node.parameters.qubits = ["Q1"]
    pass

node.machine = Quam.load()

# Local path for loading ds_raw (same as in notebook 01)
DS_RAW_PATH = os.path.join(os.getcwd(), "07b_iq_blobs_raw_adc_ds_raw.nc")

## Load saved data

In [ ]:
if os.path.exists(DS_RAW_PATH):
    dataset = xr.open_dataset(DS_RAW_PATH)
    node.results["ds_raw"] = dataset
    node.namespace["qubits"] = get_qubits(node)
    print(f"Loaded ds_raw from {DS_RAW_PATH}")
else:
    print(f"File not found: {DS_RAW_PATH}. Run notebook 01 first to execute and save.")

## Plot raw ADC traces

In [ ]:
if "ds_raw" in node.results:
    fig = plot_raw_adc_traces(node.results["ds_raw"], node.namespace["qubits"])
    node.add_node_info_subtitle(fig)
    plt.show()

## IQ blobs

In [ ]:
# if "ds_raw" in node.results:
#     qubits = node.namespace["qubits"]
#     num_qubits = len(qubits)

#     ds_demod = process_raw_dataset(node.results["ds_raw"].copy(), node)
#     node.results["ds_demod"] = ds_demod

#     ds_fit, fit_results = fit_raw_data(ds_demod, node)
#     node.results["ds_fit"] = ds_fit
#     node.results["fit_results"] = fit_results

#     fig_iq = plot_iq_blobs(ds_demod, qubits, ds_fit)
#     node.add_node_info_subtitle(fig_iq)
#     plt.show()

## IQ trajectories

In [ ]:
if "ds_raw" in node.results:
    qubits = node.namespace["qubits"]
    fig_traj = plot_iq_trajectories(node.results["ds_raw"], qubits, time_resolution_ns=150)
    node.add_node_info_subtitle(fig_traj)
    plt.show()

## SNR (Gaussian fits)

In [ ]:
# if "ds_fit" in node.results:
#     qubits = node.namespace["qubits"]
#     num_qubits = len(qubits)
#     ds_fit = node.results["ds_fit"]
#     fit_results = node.results["fit_results"]

#     cols, rows = 2, (num_qubits + 1) // 2
#     fig_snr, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows), constrained_layout=True)
#     axes = np.atleast_1d(axes).flatten()
#     snr, _, _ = fit_snr_with_gaussians(
#         fits=ds_fit, qubits=qubits, node=node, fit_results=fit_results,
#         axes=axes, plot=True,
#     )
#     for j in range(num_qubits, len(axes)):
#         axes[j].axis("off")
#     fig_snr.suptitle(node.add_node_info_subtitle())
#     plt.show()
#     node.results["snr"] = snr